# 08 - Open Data Investigator Agent

An agentic system that **searches public datasets, downloads data, analyzes patterns, and writes narrative findings** using OpenAI function-calling.

| Component | Detail |
|-----------|--------|
| Data Source | data.gov CKAN API + Census/BLS public CSVs |
| LLM | OpenAI GPT-4o-mini via function calling |
| Pattern | Question-driven exploration with tool-use loop |

## Why an Agent?

Open data investigation requires **adaptive exploration**:

1. A question like "How has housing affordability changed?" does not map to a single dataset or query.
2. The investigator must search across catalogs, evaluate dataset relevance, download and clean data, then decide what analysis to run.
3. Anomalies discovered in one dataset may prompt searching for corroborating data from another source.
4. The final output is a **narrative** -- not just numbers -- requiring synthesis across multiple analyses.

An agent can autonomously chain these steps: search -> download -> analyze -> visualize -> narrate, adapting its path based on what it finds.

In [ ]:
%pip install openai pandas requests matplotlib --quiet

In [ ]:
import json
import io
import textwrap
from typing import Optional

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import requests
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## 1 - Pre-loaded Datasets (Fallback)

The agent will attempt live API calls to data.gov. As a fallback, we pre-load representative public datasets so the notebook is runnable even if the API is slow or rate-limited.

In [ ]:
# BLS Consumer Price Index data (public domain)
# Source: https://data.bls.gov/timeseries/CUUR0000SA0
CPI_DATA = """Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
2018,247.9,248.8,249.6,250.5,251.6,251.9,252.0,252.1,252.4,252.9,252.0,251.2
2019,251.2,252.0,252.8,254.0,254.2,254.1,254.7,255.0,255.1,255.3,255.2,255.7
2020,257.0,257.4,257.0,256.4,256.3,257.2,258.7,259.1,260.2,260.4,260.2,260.5
2021,261.6,263.0,264.9,267.1,269.2,271.7,273.0,273.6,274.3,276.6,278.9,280.1
2022,281.1,284.2,287.5,289.1,292.3,296.3,296.3,296.2,296.8,298.0,298.0,296.8
2023,299.2,300.8,301.8,303.4,304.1,305.1,305.7,307.0,307.8,307.7,307.1,306.7
2024,308.4,310.3,312.2,313.5,314.1,314.2,314.5,314.8,315.3,315.6,315.5,315.6"""

# Census median household income (public domain, selected years)
INCOME_DATA = """Year,MedianHouseholdIncome,MedianHouseholdIncome_2023Dollars
2015,56516,70200
2016,59039,72100
2017,61372,73700
2018,63179,74600
2019,68703,80100
2020,67521,77400
2021,70784,76300
2022,74580,74580
2023,80610,80610"""

# Cache for downloaded datasets
dataset_cache = {}

# Pre-load fallback data
dataset_cache["cpi_annual"] = pd.read_csv(io.StringIO(CPI_DATA))
dataset_cache["median_income"] = pd.read_csv(io.StringIO(INCOME_DATA))

print(f"Pre-loaded {len(dataset_cache)} fallback datasets")

## 2 - Define Tools

In [ ]:
state = {"findings": [], "visualizations": []}


def search_data_gov(query: str) -> str:
    """Search data.gov CKAN API for datasets matching query."""
    try:
        url = "https://catalog.data.gov/api/3/action/package_search"
        resp = requests.get(url, params={"q": query, "rows": 5}, timeout=10)
        resp.raise_for_status()
        results = resp.json()["result"]["results"]
        datasets = []
        for r in results:
            csv_resources = [res for res in r.get("resources", []) if res.get("format", "").upper() == "CSV"]
            datasets.append({
                "title": r["title"],
                "name": r["name"],
                "description": (r.get("notes") or "")[:200],
                "csv_urls": [res["url"] for res in csv_resources[:2]],
                "organization": r.get("organization", {}).get("title", "Unknown"),
            })
        return json.dumps({"query": query, "count": len(datasets), "datasets": datasets})
    except Exception as e:
        # Fallback: return info about pre-loaded datasets
        return json.dumps({
            "query": query,
            "note": f"API call failed ({e}). Using pre-loaded datasets.",
            "available_cached": list(dataset_cache.keys()),
        })


def download_csv(url: str) -> str:
    """Download a CSV from URL and cache it. Returns column info and row count."""
    # Check if requesting a cached dataset by name
    if url in dataset_cache:
        df = dataset_cache[url]
    else:
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            df = pd.read_csv(io.StringIO(resp.text))
            cache_name = url.split("/")[-1].replace(".csv", "")[:30]
            dataset_cache[cache_name] = df
        except Exception as e:
            return json.dumps({"error": f"Download failed: {e}",
                               "hint": f"Available cached datasets: {list(dataset_cache.keys())}"})

    name = url if url in dataset_cache else url.split("/")[-1][:30]
    dataset_cache[name] = df

    return json.dumps({
        "name": name,
        "rows": len(df),
        "columns": list(df.columns),
        "dtypes": {c: str(df[c].dtype) for c in df.columns},
        "sample": df.head(3).to_dict(orient="records"),
    }, default=str)


def summarize_dataset(dataset_name: str) -> str:
    """Return descriptive statistics for a cached dataset."""
    df = dataset_cache.get(dataset_name)
    if df is None:
        return json.dumps({"error": f"Dataset '{dataset_name}' not found. Available: {list(dataset_cache.keys())}"})
    desc = df.describe(include="all").round(2)
    return json.dumps({
        "name": dataset_name,
        "shape": list(df.shape),
        "null_counts": df.isna().sum().to_dict(),
        "statistics": desc.to_dict(),
    }, default=str)


def find_anomalies(dataset_name: str, column: str) -> str:
    """Find anomalies in a specific column using statistical methods."""
    df = dataset_cache.get(dataset_name)
    if df is None:
        return json.dumps({"error": f"Dataset not found: {dataset_name}"})
    if column not in df.columns:
        return json.dumps({"error": f"Column '{column}' not in dataset. Columns: {list(df.columns)}"})

    s = df[column].dropna()
    result = {"column": column, "count": int(len(s))}

    if pd.api.types.is_numeric_dtype(s):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        outliers = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
        result["outlier_count"] = int(len(outliers))
        result["outlier_values"] = outliers.head(10).tolist()
        # Year-over-year changes if data looks like a time series
        if len(s) > 2:
            pct_changes = s.pct_change().dropna()
            result["max_pct_change"] = round(float(pct_changes.max()) * 100, 2)
            result["min_pct_change"] = round(float(pct_changes.min()) * 100, 2)
            result["mean_pct_change"] = round(float(pct_changes.mean()) * 100, 2)
    else:
        vc = s.value_counts()
        result["unique_values"] = int(vc.shape[0])
        result["most_common"] = vc.head(5).to_dict()
        result["least_common"] = vc.tail(3).to_dict()

    return json.dumps(result, default=str)


def create_visualization(dataset_name: str, chart_type: str, columns: list) -> str:
    """Create a chart and save it. Returns the file path."""
    df = dataset_cache.get(dataset_name)
    if df is None:
        return json.dumps({"error": f"Dataset not found: {dataset_name}"})

    fig, ax = plt.subplots(figsize=(10, 5))
    try:
        if chart_type == "line" and len(columns) >= 2:
            for col in columns[1:]:
                ax.plot(df[columns[0]], df[col], marker="o", label=col)
            ax.set_xlabel(columns[0])
            ax.legend()
        elif chart_type == "bar" and len(columns) >= 2:
            ax.bar(df[columns[0]].astype(str), df[columns[1]])
            ax.set_xlabel(columns[0])
            ax.set_ylabel(columns[1])
            plt.xticks(rotation=45)
        elif chart_type == "scatter" and len(columns) >= 2:
            ax.scatter(df[columns[0]], df[columns[1]], alpha=0.6)
            ax.set_xlabel(columns[0])
            ax.set_ylabel(columns[1])
        elif chart_type == "hist":
            ax.hist(df[columns[0]].dropna(), bins=20, edgecolor="black")
            ax.set_xlabel(columns[0])
        else:
            return json.dumps({"error": f"Unsupported chart_type '{chart_type}' with {len(columns)} columns"})

        ax.set_title(f"{chart_type.title()} Chart: {', '.join(columns)}")
        plt.tight_layout()
        fname = f"viz_{len(state['visualizations'])+1}_{chart_type}.png"
        fig.savefig(fname, dpi=100)
        plt.close(fig)
        state["visualizations"].append(fname)
        return json.dumps({"chart_file": fname, "chart_type": chart_type, "columns": columns})
    except Exception as e:
        plt.close(fig)
        return json.dumps({"error": str(e)})


def write_finding(analysis_result: str) -> str:
    """Record an analytical finding in the investigation report."""
    finding = {
        "id": f"F-{len(state['findings'])+1:02d}",
        "content": analysis_result,
    }
    state["findings"].append(finding)
    return json.dumps({"recorded": finding["id"], "total_findings": len(state["findings"])})

## 3 - Tool Dispatcher & Schemas

In [ ]:
TOOL_MAP = {
    "search_data_gov": search_data_gov,
    "download_csv": download_csv,
    "summarize_dataset": summarize_dataset,
    "find_anomalies": find_anomalies,
    "create_visualization": create_visualization,
    "write_finding": write_finding,
}


def dispatch_tool(name: str, arguments: dict) -> str:
    fn = TOOL_MAP.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return json.dumps({"error": str(e)})


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_data_gov",
            "description": "Search data.gov for datasets matching a query. Returns dataset titles, descriptions, and CSV URLs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query for data.gov"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "download_csv",
            "description": "Download a CSV from URL or load a cached dataset by name. Returns column info and sample rows.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "CSV URL or cached dataset name (e.g., 'cpi_annual', 'median_income')"}
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_dataset",
            "description": "Return descriptive statistics for a cached dataset.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_name": {"type": "string", "description": "Name of the cached dataset"}
                },
                "required": ["dataset_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "find_anomalies",
            "description": "Find outliers and anomalies in a specific column of a cached dataset.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_name": {"type": "string", "description": "Name of the cached dataset"},
                    "column": {"type": "string", "description": "Column to analyze"}
                },
                "required": ["dataset_name", "column"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_visualization",
            "description": "Create a chart (line, bar, scatter, hist) from a cached dataset.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_name": {"type": "string", "description": "Name of the cached dataset"},
                    "chart_type": {"type": "string", "enum": ["line", "bar", "scatter", "hist"]},
                    "columns": {"type": "array", "items": {"type": "string"}, "description": "Column names to plot. First column is x-axis."}
                },
                "required": ["dataset_name", "chart_type", "columns"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_finding",
            "description": "Record an analytical finding in the investigation report.",
            "parameters": {
                "type": "object",
                "properties": {
                    "analysis_result": {"type": "string", "description": "The finding to record, including evidence and interpretation"}
                },
                "required": ["analysis_result"]
            }
        }
    }
]

## 4 - Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
You are an Open Data Investigator Agent. You answer research questions using public datasets.

Pre-loaded datasets available: {cached}

Workflow:
1. Understand the user's question.
2. Search data.gov for relevant datasets. Also consider the pre-loaded datasets.
3. Download promising CSVs (or use cached datasets by passing their name as the url).
4. Summarize the datasets to understand their structure.
5. Find anomalies in key columns.
6. Create visualizations to illustrate trends.
7. Write findings as you go using write_finding.
8. After thorough analysis, write a final narrative summary and say DONE.

Be data-driven: cite specific numbers. Create at least 2 visualizations.
""")


def run_agent(question: str, max_turns: int = 25):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(cached=list(dataset_cache.keys()))},
        {"role": "user", "content": question}
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"\n[Turn {turn+1}] Agent says:\n{msg.content[:600]}")
            if msg.content and "DONE" in msg.content.upper():
                break
            continue

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"[Turn {turn+1}] Tool: {tc.function.name}({json.dumps(args)[:100]})")
            result = dispatch_tool(tc.function.name, args)
            print(f"         Result: {result[:200]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return messages

In [ ]:
question = (
    "How has the cost of living in the US changed from 2018 to 2024? "
    "Compare CPI trends with median household income to assess whether "
    "wages have kept pace with inflation. Identify any notable anomalies."
)
print(f"Investigation question:\n{question}\n")
conversation = run_agent(question)

## 5 - Results Analysis

In [ ]:
print(f"Findings recorded: {len(state['findings'])}")
print(f"Visualizations created: {len(state['visualizations'])}")
print(f"Datasets in cache: {list(dataset_cache.keys())}")

print("\n" + "=" * 60)
print("INVESTIGATION FINDINGS")
print("=" * 60)
for f in state["findings"]:
    print(f"\n[{f['id']}] {f['content']}")

In [ ]:
# Display saved visualizations
from IPython.display import Image, display
import os

for viz in state["visualizations"]:
    if os.path.exists(viz):
        print(f"\n--- {viz} ---")
        display(Image(filename=viz))
    else:
        print(f"Visualization file not found: {viz}")

In [ ]:
# Show the final narrative from the agent
for m in reversed(conversation):
    content = m.content if hasattr(m, 'content') else m.get('content')
    if content and "DONE" in (content or "").upper():
        print("\nFinal Narrative:")
        print(content)
        break

## 6 - Key Takeaways

1. **Question-driven exploration**: The agent started with a question and autonomously decided which datasets and analyses were relevant.
2. **Graceful fallback**: When API calls fail, the agent adapts by using pre-loaded data.
3. **Multi-source synthesis**: The agent combined CPI and income data to answer a question neither dataset answers alone.
4. **Narrative output**: Findings are recorded incrementally and synthesized into a readable report.
5. **Extensibility**: Adding new data sources (FRED, World Bank) just requires new search/download functions.